In [1]:
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.core import Settings
from llama_index.readers.file import PagedCSVReader
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core.ingestion import IngestionPipeline
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, ServiceContext, load_index_from_storage
from llama_index.llms.llama_cpp import LlamaCPP
from langchain_community.chat_models.llamacpp import ChatLlamaCpp
from llama_index.llms.llama_cpp.llama_utils import (messages_to_prompt,completion_to_prompt,)
from llama_index.core.node_parser import SentenceWindowNodeParser
import faiss
import os, sys
import pandas as pd
from dotenv import load_dotenv
from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler
from langchain.document_loaders import CSVLoader
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader
from langchain.embeddings import OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
__import__('pysqlite3')
import pysqlite3
sys.modules['sqlite3'] = sys.modules["pysqlite3"]

/home/ubuntu/LLM/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Llamaindex global settings for llm and embeddings
EMBED_DIMENSION=384 # all=M<iniLM-L6-v2
#EMBED_DIMENSION = 768 # all-mpnet-base-v2
# Configure your local TinyLlama GGUF model
llm = ChatLlamaCpp(
    model_path="../models/llm/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",  # Path to your GGUF file
    temperature=0.1,
    max_new_tokens=2048,
    n_gpu_layers=1,
    n_ctx=4082,
    n_batch=512, 
    # kwargs to pass to __call__()
    generate_kwargs={},
     # Adjust based on model's max context
    model_kwargs={},
    f16_kv=True,
    # transform inputs into Llama2 format
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True # Enable GPU acceleration
)
# hf = SentenceTransformer("/home/ubuntu/LLM/models/embedding/all-MiniLM-L6-v2")
# Settings.embed_model = HuggingFaceEmbedding(
#     model_name="../models/embedding/all-mpnet-base-v2/",  # Download GGUF file first
#     device="cuda"
# )


ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA A10G, compute capability 8.6, VMM: yes


llama_model_load_from_file_impl: using device CUDA0 (NVIDIA A10G) - 22259 MiB free
llama_model_loader: loaded meta data with 23 key-value pairs and 201 tensors from ../models/llm/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = tinyllama_tinyllama-1.1b-chat-v1.0
llama_model_loader: - kv   2:                       llama.context_length u32              = 2048
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 5632
llama_model_loader: - kv   6:                 ll

In [6]:
file_path = ('/home/ubuntu/LLM/data/Tourist_attractions_ROI_formatted.csv') # insert the path of the csv file
data = pd.read_csv(file_path)

In [7]:
loader = CSVLoader('/home/ubuntu/LLM/data/Tourist_attractions_ROI_formatted.csv', encoding="windows-1252")
documents = loader.load()

In [8]:
embedding_function = SentenceTransformerEmbeddings(model_name="/home/ubuntu/LLM/models/embedding/all-MiniLM-L6-v2")

/tmp/ipykernel_19785/3680649334.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="/home/ubuntu/LLM/models/embedding/all-MiniLM-L6-v2")


In [9]:
db = Chroma.from_documents(documents, embedding_function)
retriever = db.as_retriever()

In [10]:
template = """You are a tour guide of the city of Belem, in the state of Pará in the country of Brazil. You must provide directions and respond to basic questions about touristic places in Belem. You must be very precise and provide the best possible answer to the user, in a concise way.
You are given a context with information about the touristic places in Belem. You must use this information to answer the user's question. If you do not know the answer, say "I don't know".
Context: {context}

Question: {question}

"""

prompt = ChatPromptTemplate.from_template(template)

In [11]:
from langchain.chains import RetrievalQA
# Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=db.as_retriever())# User query
query = "What is the address of the clock square?"# Get the response
result = qa_chain.run(query)
print(result)

/tmp/ipykernel_19785/720040687.py:5: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain.run(query)
Llama.generate: 2 prefix-match hit, remaining 2452 prompt tokens to eval


s:
<a/yes:s:t:0:s:
and:s:
the:s:n:s:s:iq:e:s tow to the:ss:data:s forumsphere:13: asearch:st,ph,pars andvoice, or:a:ãning.
s:s,singing.s,s.
your <and:thean'informationalicleptântsage:s:certainnancesionsans,t,s,s,ss, hows whatsange.nsineppings:mentingmentsionsionsories:r:sure:sw:or:ss:information:tenning,euants and <what. Buts. The a:andums. â, buts:
<meanings orppingsksheeteingdingaclesionsanseshesuresuesumsumsumsumsionshsingseranceographymentnsppings'informationmentningningionments:ss,tsterningningningningmentsicsumsgedgingments as the information’information:estdingnsions data:words:informationatoryetningmentning questioninggsningningmenteu, are, addressting.

llama_perf_context_print:        load time =     388.64 ms
llama_perf_context_print: prompt eval time =     709.54 ms /  2452 tokens (    0.29 ms per token,  3455.75 tokens per second)
llama_perf_context_print:        eval time =    6597.16 ms /   255 runs   (   25.87 ms per token,    38.65 tokens per second)
llama_perf_context_print:       total time =    7751.10 ms /  2707 tokens


s:
<a/yes:s:t:0:s:
and:s:
the:s:n:s:s:iq:e:s tow to the:ss:data:s forumsphere:13: asearch:st,ph,pars andvoice, or:a:ãning.
s:s,singing.s,s.
your <and:thean'informationalicleptântsage:s:certainnancesionsans,t,s,s,ss, hows whatsange.nsineppings:mentingmentsionsionsories:r:sure:sw:or:ss:information:tenning,euants and <what. Buts. The a:andums. â, buts:
<meanings orppingsksheeteingdingaclesionsanseshesuresuesumsumsumsumsionshsingseranceographymentnsppings'informationmentningningionments:ss,tsterningningningningmentsicsumsgedgingments as the information’information:estdingnsions data:words:informationatoryetningmentning questioninggsningningmenteu, are, addressting.


In [12]:
chain = (

    {"context": retriever, "question": RunnablePassthrough()}

    | prompt

    | llm

    | StrOutputParser()

)

In [13]:
res = chain.invoke("Which state of Brazil is the Clock Square located?")

Llama.generate: 2 prefix-match hit, remaining 2848 prompt tokens to eval


information,lning inmentumsationsments,owards.or, language, language or information language language language language languageesnsingning language, bestest, language, or,guideumscriptsmltting,l,lning.menticulnds asging asnsksmingns, as as,estment, as,l,1, as,l,mat,l,language,l, lat01,parnesment_information_2479_gu_l5_latding_m\materials_l_l_l_l13lmentment.l.lnd1tindrn608l111011matning\desestininglndl:parns pare1 and par,parest,l,par,lat111lparsparspar,l to,lat,rn latm.prov lrn2rn3rn17l to to to to114est ro015611,1,mat,par,191,par,par pars1s c1010111181000110pars_tiest

llama_perf_context_print:        load time =     388.64 ms
llama_perf_context_print: prompt eval time =     831.87 ms /  2848 tokens (    0.29 ms per token,  3423.60 tokens per second)
llama_perf_context_print:        eval time =    6846.10 ms /   255 runs   (   26.85 ms per token,    37.25 tokens per second)
llama_perf_context_print:       total time =    8060.92 ms /  3103 tokens


In [14]:
res

'information,lning inmentumsationsments,owards.or, language, language or information language language language language languageesnsingning language, bestest, language, or,guideumscriptsmltting,l,lning.menticulnds asging asnsksmingns, as as,estment, as,l,1, as,l,mat,l,language,l, lat01,parnesment_information_2479_gu_l5_latding_m\\materials_l_l_l_l13lmentment.l.lnd1tindrn608l111011matning\\desestininglndl:parns pare1 and par,parest,l,par,lat111lparsparspar,l to,lat,rn latm.prov lrn2rn3rn17l to to to to114est ro015611,1,mat,par,191,par,par pars1s c1010111181000110pars_tiest'